# InMobi EDA — ClickHouse Cloud

House rule: aggregation/analysis happens in ClickHouse SQL, not pandas. `ch.query_df()` runs a query and returns the (already small/aggregated) result as a DataFrame for plotting — never pull raw `ad_events` rows here.

In [1]:
from ch import query_df
import plotly.express as px

## Data dictionary — `inmobi` schema (ClickHouse)

Funnel: **Request → Fill → Impression → Click**, revenue earned on impressions. Full source: [`InMobi/data/ddl.sql`](../data/ddl.sql), [`InMobi/metrics_glossary.md`](../metrics_glossary.md).

### `ad_events` (fact table, ~9M rows, 5 weeks, `PARTITION BY toDate(event_time)`, `ORDER BY (event_time, app_id)`)

| Column | Type | Notes |
|---|---|---|
| `event_time` | `DateTime64(3)` | |
| `app_id` | `String` | FK → `apps.app_id` |
| `geo_device_id` | `String` | FK → `geo_device.geo_device_id` |
| `advertiser_id` | `String` | FK → `advertisers.advertiser_id`; **empty string on unfilled requests** — inner-join carefully |
| `ad_format` | `LowCardinality(String)` | `banner, interstitial, native, rewarded, video` |
| `is_filled` | `UInt8` | 1 if request was filled |
| `is_impression` | `UInt8` | 1 if fill rendered as an impression |
| `is_click` | `UInt8` | 1 if impression was clicked |
| `revenue` | `Float64` | earned on impressions |

### Dimension tables

| Table | Rows | Columns |
|---|---|---|
| `apps` | 2,000 | `app_id`, `category` (`gaming, social, entertainment, news, ecommerce, utility, finance`), `publisher_tier` (`tier_1, tier_2, tier_3`) |
| `advertisers` | 500 | `advertiser_id`, `vertical` (`gaming, ecommerce, finance, travel, entertainment, auto, cpg`), `campaign_type` (`CPM, CPC, CPI`) |
| `geo_device` | 5,000 | `geo_device_id`, `region` (`NAM, EU, APAC, LATAM, MEA` — NAM not NA), `country` (US, CA, UK, DE, IN, JP, BR, MX, ZA, AE), `device_model`, `os_version` (`iOS 16.4/17.2/17.5/18.1`, `Android 12/13/14/15`) |

### Core metrics (always `sum / sum` over the group — never row- or day-averaged)

| Metric | Formula |
|---|---|
| Requests | `count(*)` |
| Fill rate | `sum(is_filled) / count(*)` |
| Render rate | `sum(is_impression) / sum(is_filled)` |
| CTR | `sum(is_click) / sum(is_impression)` |
| Revenue | `sum(revenue)` |
| eCPM | `sum(revenue) / sum(is_impression) * 1000` |
| RPR (revenue per request) | `sum(revenue) / count(*)` |

Revenue identity: `Revenue ≈ Requests × Fill rate × eCPM / 1000` — walk this to find which factor (volume/fill/price) moved before slicing by dimension.

Data has daily + weekly seasonality, a slow growth trend, and noise — compare against a like-for-like baseline (same hour-of-day, same day-of-week, trailing weeks), not a flat average.

### The funnel in plain English (Request → Fill → Impression → Click)

Each step is a subset of the one before it — `is_filled`, `is_impression`, `is_click` are cumulative flags (impression implies filled, click implies impression).

- **Request** — the app asks an ad network "do you have an ad for me right now?" Every row in `ad_events` starts as one of these — the top-of-funnel opportunity.
- **Fill** (`is_filled`) — the ad network says yes and returns an actual ad creative. Not every request gets filled — sometimes no advertiser wants that impression, so the request goes unanswered. Fill rate is "of the times we asked, how often did someone offer an ad."
- **Impression** (`is_impression`) — the ad was filled *and then actually shown on screen*. This is a separate step from fill because things can go wrong in between — the user backs out of the app, a network timeout, a video ad fails to load. Render rate is "of the ads we had, how many actually rendered." **Revenue is earned on impressions, not fills.**
- **Click** (`is_click`) — the user tapped the ad after seeing it. CTR is "of the ads shown, how many got tapped." An engagement/quality signal, not a direct revenue driver here — this dataset is CPM-priced (cost per thousand impressions), not cost-per-click.

So a single row can land anywhere on the funnel: requested-but-not-filled (no ad available), filled-but-not-impression (ad received but didn't render), impression-but-no-click (shown but ignored), or all the way through to a click.

### Metrics glossary, explained

All ratio metrics are **sum / sum over the group**, never an average of per-row or per-day ratios — averaging ratios across uneven-sized groups silently corrupts rollups (e.g. a day with 10 requests and 100% fill rate would wrongly pull as much weight as a day with 1M requests and 50% fill rate).

- **Requests** — `count(*)`. Raw volume: how many ad opportunities existed. The top-of-funnel number everything else is a fraction of. A drop here means fewer people are opening apps / triggering ad slots, not a monetization problem per se.

- **Fills** — `sum(is_filled)`. Count of requests that got an ad back.

- **Fill rate** — `sum(is_filled) / count(*)`. What fraction of ad opportunities the network could actually monetize. A fill-rate drop usually means supply-demand mismatch — not enough advertiser demand for that segment (app/geo/device), a broken ad-network integration, or a pricing floor set too high for that inventory.

- **Impressions** — `sum(is_impression)`. Count of ads that were filled *and* rendered on screen.

- **Render rate** — `sum(is_impression) / sum(is_filled)`. Of the ads we won, how many actually made it to the screen. This is the "fill → impression" leak — usually a technical/latency issue (slow creative load, user navigates away, SDK bug) rather than a demand issue. Near 100% is healthy; a drop here points at the client/app side, not the ad marketplace.

- **Clicks** — `sum(is_click)`. Count of impressions the user tapped.

- **CTR (click-through rate)** — `sum(is_click) / sum(is_impression)`. Engagement quality: are people interested in what's being shown. Useful diagnostic signal but — in this CPM-priced dataset — not itself a revenue driver; it's a sibling metric to check creative/targeting quality, not a factor in the revenue identity below.

- **Revenue** — `sum(revenue)`. Money earned, attributed to impressions (not fills, not clicks). The bottom-line number stakeholders actually care about.

- **eCPM (effective cost per mille)** — `sum(revenue) / sum(is_impression) * 1000`. The *price* — how much revenue each 1,000 impressions generates. An eCPM drop with stable fill/render rate points at a pricing problem: advertiser demand shifted to cheaper campaigns, an auction floor was misconfigured, or a high-value advertiser vertical churned.

- **RPR (revenue per request)** — `sum(revenue) / count(*)`. All-in efficiency per opportunity — folds together volume, fill, render, and price into one number. Useful as a single top-line health check, but not diagnostic on its own: an RPR drop could be caused by any of the upstream factors, which is why you decompose it via the revenue identity instead of staring at RPR directly.

**Why decompose via the revenue identity:** `Revenue ≈ Requests × Fill rate × eCPM / 1000`. When revenue moves, this identity tells you *which lever* moved — more/fewer opportunities (requests), worse/better monetization rate (fill rate), or a pricing shift (eCPM) — before you go slicing by dimension (app, geo, device, advertiser) to find *where*. Chasing a revenue dip by dimension-slicing without first walking this identity risks finding a segment that correlates but isn't actually the causal factor.

### What the loaded data actually looks like (queried, not from docs)

- **Date range:** 2026-06-01 to 2026-07-05 (34 days spanning the documented "5 weeks").
- **Global funnel:** 9.00M requests → 7.03M fills (**78.1%** fill rate) → 6.89M impressions (**98.0%** render rate) → 74.9K clicks (**1.09%** CTR). Revenue $17,020 → **eCPM $2.47**, **RPR $0.00189**.
- **`advertiser_id` empty confirms exactly on unfilled rows** — 1,972,090 empty `advertiser_id` = 1,972,090 rows with `is_filled = 0`, and 0 empty among the 7,027,910 filled rows. No `app_id`/`geo_device_id` are empty. Safe to treat `advertiser_id = ''` as the unfilled marker.
- **`ad_format` mix is uneven, not uniform** across its 5 values: banner 3.13M (35%) > native 2.35M (26%) > interstitial 1.56M (17%) > video 1.17M (13%) > rewarded 0.78M (9%) — worth weighting when comparing formats for anomaly localization.
- **Dimension table sizes match the docs exactly**: `apps` 2,000, `advertisers` 500, `geo_device` 5,000 — `gaming`/`tier_2` is the single largest app segment (245 apps); `travel`/`CPM` is the largest advertiser segment (48).
- **`geo_device` region skew**: APAC (1,418) and NAM (1,359) each carry ~4x the country breadth of MEA (450, 3 countries) — expect thinner per-segment sample sizes in MEA/LATAM baselines, which raises noise in any per-region z-score.
- **Weekly seasonality confirmed in raw counts**: `toDayOfWeek` requests taper from ~1.38M (Mon-Wed) down to 1.32M (Fri), 1.12M (Sat), 1.05M (Sun) — a real ~24% weekday-to-Sunday drop, exactly the seasonality the glossary warns not to alarm on.

In [2]:
query_df("""
SELECT min(event_time) AS start, max(event_time) AS end,
       dateDiff('day', min(event_time), max(event_time)) AS days,
       count() AS requests, sum(is_filled) AS fills, sum(is_filled) / count() AS fill_rate,
       sum(is_impression) AS impressions, sum(is_impression) / sum(is_filled) AS render_rate,
       sum(is_click) AS clicks, sum(is_click) / sum(is_impression) AS ctr,
       sum(revenue) AS total_revenue, sum(revenue) / sum(is_impression) * 1000 AS ecpm,
       sum(revenue) / count() AS rpr
FROM inmobi.ad_events
""")

,start,end,days,requests,fills,fill_rate,impressions,render_rate,clicks,ctr,total_revenue,ecpm,rpr
0,2026-06-01,2026-07-05 23:59:59,34,9000000,7027910,0.780879,6887058,0.979958,74940,0.010881,17020.364187,2.471355,0.001891


In [3]:
query_df("""
SELECT table, sum(rows) AS rows
FROM system.parts
WHERE database = 'inmobi' AND active
GROUP BY table ORDER BY table
""")

,table,rows
0,ad_events,9000000
1,advertisers,500
2,anomalies,349
3,apps,2000
4,detection_config,50
5,geo_device,5000
6,incidents,11
7,metric_noise_baseline,7
8,metric_seasonal_ewma_state,14
9,metric_zr_hourly,3432


## Hourly rollup — top-level shape

In [4]:
df = query_df("""
SELECT
    toStartOfHour(event_time) AS hour,
    count() AS requests,
    sum(is_filled) / count() AS fill_rate,
    sum(is_impression) / nullif(sum(is_filled), 0) AS render_rate,
    sum(is_click) / nullif(sum(is_impression), 0) AS ctr,
    sum(revenue) AS total_revenue,
    sum(revenue) / nullif(sum(is_impression), 0) * 1000 AS ecpm,
    sum(revenue) / count() AS rpr
FROM inmobi.ad_events
GROUP BY hour
ORDER BY hour
""")
df.head()

,hour,requests,fill_rate,render_rate,ctr,total_revenue,ecpm,rpr
0,2026-06-01 00:00:00,7368,0.787324,0.977245,0.008467,14.153259,2.496606,0.001921
1,2026-06-01 01:00:00,7769,0.780667,0.983677,0.010560,14.704443,2.464707,0.001893
2,2026-06-01 02:00:00,7966,0.783329,0.980288,0.010790,15.406953,2.518711,0.001934
3,2026-06-01 03:00:00,8854,0.792071,0.980322,0.010327,17.174618,2.498126,0.001940
4,2026-06-01 04:00:00,9709,0.788959,0.980287,0.010388,18.467860,2.459430,0.001902


In [5]:
px.line(df, x="hour", y="requests", title="Requests per hour")

In [6]:
px.line(df, x="hour", y="fill_rate", title="Fill rate per hour")

In [7]:
px.line(df, x="hour", y="render_rate", title="Render rate per hour")

In [8]:
px.line(df, x="hour", y="ctr", title="CTR per hour")

In [9]:
px.line(df, x="hour", y="total_revenue", title="Revenue per hour")

In [10]:
px.line(df, x="hour", y="ecpm", title="eCPM per hour")

In [11]:
px.line(df, x="hour", y="rpr", title="RPR (revenue per request) per hour")

### Baseline reference — global average per metric

A single "what does normal look like" reference table before any anomaly detection runs — the whole-dataset `sum/sum` average for each metric, computed the same way every ratio metric must be (never row- or day-averaged, per the metrics glossary). This is the number every later z-score, lift, and % delta in this notebook is implicitly being compared against.

In [12]:
import pandas as pd

baseline_reference = query_df("""
SELECT
    count() AS requests,
    sum(is_filled) / count() AS fill_rate,
    sum(is_impression) / nullif(sum(is_filled), 0) AS render_rate,
    sum(is_click) / nullif(sum(is_impression), 0) AS ctr,
    sum(revenue) AS total_revenue,
    sum(revenue) / nullif(sum(is_impression), 0) * 1000 AS ecpm,
    sum(revenue) / count() AS rpr
FROM inmobi.ad_events
""")

BASELINE_FMT = {
    "requests": "{:,.0f}", "fill_rate": "{:.2%}", "render_rate": "{:.2%}",
    "ctr": "{:.3%}", "total_revenue": "${:,.2f}", "ecpm": "${:.3f}", "rpr": "${:.5f}",
}
pd.DataFrame({
    "global baseline (sum/sum)": [BASELINE_FMT[c].format(baseline_reference[c].iloc[0]) for c in baseline_reference.columns]
}, index=baseline_reference.columns)

,global baseline (sum/sum)
requests,"9,000,000"
fill_rate,78.09%
render_rate,98.00%
ctr,1.088%
total_revenue,"$17,020.36"
ecpm,$2.471
rpr,$0.00189


## Calendar-shift comparison — one selected ISO week vs the previous week

This mirrors Datadog's `calendar_shift(metric, "-1w", "UTC")`: choose one displayed ISO week, then draw that week's actual values against the previous calendar week's values shifted onto the displayed timestamps. Each chart has a week selector; it never draws several rolling comparisons on one continuous timeline, which would make a real incident appear again one week later. The shifted trace is dashed and its hover shows the value's original timestamp.

Weeks are anchored **Monday–Sunday in UTC** via `toStartOfWeek(hour, 1)`. The first calendar week has no prior-week comparator and is excluded. A divergence is evidence about the displayed week only; if the dashed comparator contains a prior incident, its hover makes that contaminated reference explicit.

In [13]:
import plotly.graph_objects as go

METRICS = ["requests", "fill_rate", "render_rate", "ctr", "total_revenue", "ecpm", "rpr"]

lag_cols = ",\n    ".join(
    f"lagInFrame({m}, 168) OVER (ORDER BY hour ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING) AS {m}_lag1w"
    for m in METRICS
)

wow = query_df(f"""
WITH hourly AS (
    SELECT
        toStartOfHour(event_time) AS hour,
        toStartOfWeek(event_time, 1) AS week_start,
        count() AS requests,
        sum(is_filled) / count() AS fill_rate,
        sum(is_impression) / nullif(sum(is_filled), 0) AS render_rate,
        sum(is_click) / nullif(sum(is_impression), 0) AS ctr,
        sum(revenue) AS total_revenue,
        sum(revenue) / nullif(sum(is_impression), 0) * 1000 AS ecpm,
        sum(revenue) / count() AS rpr
    FROM inmobi.ad_events
    GROUP BY hour, week_start
),
lagged AS (
    SELECT hour, week_start,
    lagInFrame(hour, 168) OVER (ORDER BY hour ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING) AS prior_hour,
    {", ".join(METRICS)},
    {lag_cols}
    FROM hourly
)
SELECT * FROM lagged
WHERE week_start > (SELECT min(week_start) FROM hourly)
ORDER BY hour
""")


def calendar_shift_plot(metric, title):
    weeks = sorted(pd.to_datetime(wow["week_start"].unique()))
    fig = go.Figure()
    titles = []
    for i, week in enumerate(weeks):
        d = wow[pd.to_datetime(wow["week_start"]) == week].copy()
        current_end = week + pd.Timedelta(days=6)
        prior_start, prior_end = week - pd.Timedelta(weeks=1), current_end - pd.Timedelta(weeks=1)
        chart_title = (f"{title} — {week:%d %b}–{current_end:%d %b %Y} "
                       f"vs calendar_shift(-1w, UTC): {prior_start:%d %b}–{prior_end:%d %b %Y}")
        titles.append(chart_title)
        visible = i == len(weeks) - 1
        fig.add_trace(go.Scatter(
            x=d["hour"], y=d[metric], mode="lines", name="actual", visible=visible,
            line=dict(color="#636EFA", width=2.5),
            hovertemplate="Displayed: %{x|%Y-%m-%d %H:%M UTC}<br>Actual: %{y:,.6g}<extra></extra>",
        ))
        fig.add_trace(go.Scatter(
            x=d["hour"], y=d[f"{metric}_lag1w"], mode="lines",
            name="calendar_shift(-1w, UTC)", visible=visible,
            line=dict(color="#EF553B", width=1.8, dash="dot"), customdata=d[["prior_hour"]],
            hovertemplate=("Displayed: %{x|%Y-%m-%d %H:%M UTC}<br>"
                           "Original: %{customdata[0]|%Y-%m-%d %H:%M UTC}<br>"
                           "Shifted value: %{y:,.6g}<extra></extra>"),
        ))
    buttons = []
    for i, week in enumerate(weeks):
        trace_visibility = [False] * (2 * len(weeks))
        trace_visibility[2 * i:2 * i + 2] = [True, True]
        buttons.append(dict(
            label=f"{week:%d %b %Y}", method="update",
            args=[{"visible": trace_visibility}, {"title": titles[i]}],
        ))
    fig.update_layout(
        title=titles[-1], xaxis_title="displayed hour (UTC)", yaxis_title="value",
        legend_title_text="series", hovermode="x unified",
        updatemenus=[dict(buttons=buttons, direction="down", x=0.0, y=1.16,
                          xanchor="left", yanchor="top", showactive=True)],
        margin=dict(t=120),
    )
    return fig


wow.head()

,hour,week_start,prior_hour,requests,fill_rate,render_rate,ctr,total_revenue,ecpm,rpr,requests_lag1w,fill_rate_lag1w,render_rate_lag1w,ctr_lag1w,total_revenue_lag1w,ecpm_lag1w,rpr_lag1w
0,2026-06-08 00:00:00,2026-06-08,2026-06-01 00:00:00,7695,0.779077,0.979316,0.009879,14.734460,2.509702,0.001915,7368,0.787324,0.977245,0.008467,14.153259,2.496606,0.001921
1,2026-06-08 01:00:00,2026-06-08,2026-06-01 01:00:00,7820,0.782609,0.980065,0.010337,14.801671,2.467768,0.001893,7769,0.780667,0.983677,0.010560,14.704443,2.464707,0.001893
2,2026-06-08 02:00:00,2026-06-08,2026-06-01 02:00:00,8256,0.791182,0.981476,0.010451,15.744825,2.455908,0.001907,7966,0.783329,0.980288,0.010790,15.406953,2.518711,0.001934
3,2026-06-08 03:00:00,2026-06-08,2026-06-01 03:00:00,8982,0.790804,0.980853,0.011052,17.426536,2.501297,0.001940,8854,0.792071,0.980322,0.010327,17.174618,2.498126,0.001940
4,2026-06-08 04:00:00,2026-06-08,2026-06-01 04:00:00,9967,0.781278,0.982278,0.011766,19.006621,2.484850,0.001907,9709,0.788959,0.980287,0.010388,18.467860,2.459430,0.001902


In [14]:
calendar_shift_plot("requests", "Requests")

In [15]:
calendar_shift_plot("fill_rate", "Fill rate")

In [16]:
calendar_shift_plot("render_rate", "Render rate")

In [17]:
calendar_shift_plot("ctr", "CTR")

In [18]:
calendar_shift_plot("total_revenue", "Revenue")

In [19]:
calendar_shift_plot("ecpm", "eCPM")

In [20]:
calendar_shift_plot("rpr", "RPR (revenue per request)")

## Fresh anomaly sweep — active 3-method ensemble, computed entirely in ClickHouse SQL

Ignoring the persisted `inmobi.anomalies` rows for this independent validation — every number below comes from SQL run fresh against `inmobi.ad_events`, using the same definitions and thresholds as `apps/detection-service`. Deliberately **not** using ClickHouse's native `seriesDecomposeSTL` for the seasonal piece — that function wants one evenly-spaced series, but this needs per-(day-of-week, hour-of-day) cells plus a separate trailing-trend detrending step for volume metrics, which the hand-rolled window-function version handles directly.

**Three active methods, OR'd together** (a hit from any one independently qualifies its hour or completed day):

1. **`trend_seasonal`** (all 7 metrics) — observed vs. a trailing seasonal baseline: mean/variance per `(dow, hod)` cell computed over the **trailing 4 weeks only** (`ROWS BETWEEN 4 PRECEDING AND 1 PRECEDING` — causal, never future data, unlike this notebook's earlier leave-one-out draft). Volume metrics (`requests`, `total_revenue`) are first divided by a trailing 7-day daily-total moving average so growth isn't misread as a jump. Variance is **shrunk toward each metric's pooled residual variance** (`(n·var_cell + 3·var_pooled)/(n+3)`) since a 2-4 point trailing stddev is unstable on its own. Per-metric thresholds match the engine: requests 2.5, revenue 2.1, fill rate 2.2, render rate 4.0, CTR 4.0, eCPM 3.3, and RPR 3.4.
2. **`proportion`** (`fill_rate`, `render_rate`, `ctr` — count-over-count ratios only) — a two-proportion z-test using the hour's own numerator/denominator counts against the trailing baseline rate: `z = (observed_rate − p0) / sqrt(p0·(1−p0)/n)`. Count-aware, so it's far more sensitive at InMobi's hourly volume than `trend_seasonal` on the same ratio. Thresholds: fill rate 3.0, render rate 4.0, CTR 4.0.
3. **`day_level`** — pools each day's 24 `trend_seasonal` residuals into a one-sample t-test: `day_t = mean(zr) × sqrt(24)`. Catches a mild, persistent whole-day shift too small for any single hour to clear its own threshold. Threshold `|day_t| > 3.0`.

**Dual-grain qualification policy:** every metric is evaluated at both grains, and each statistical unit qualifies independently. An hour qualifies when an active hourly method (`trend_seasonal` or `proportion`) crosses its configured threshold; a completed day qualifies when its `day_level` score crosses its threshold. There is no extra `3-of-4`, rolling four-hour, eight-hour, or two-day persistence gate. CTR and render rate are reporting-only: their flags remain available for secondary correlation but cannot open an incident. Adverse qualified units from incident-enabled metrics become reportable candidates; positive units remain informational. Adjacent/overlapping qualified units are grouped afterward only to describe one incident window, never to decide whether they qualify.

In [21]:
import pandas as pd

# Shared seasonal-residual CTE ("zr"), reused (inline, self-contained) by trend_seasonal,
# and day_level below — matches the active pipeline's seasonal-residual shape, computed
# fresh from ad_events rather than the old pipeline's materialized metric_zr_hourly table.
ZR_CTE = """
WITH hourly AS (
  SELECT toStartOfHour(event_time) AS hour_ts, toDate(event_time) AS d,
    toDayOfWeek(event_time) AS dow, toHour(event_time) AS hod,
    count() AS requests, sum(is_filled) AS fills, sum(is_impression) AS impressions,
    sum(is_click) AS clicks, sum(revenue) AS revenue
  FROM inmobi.ad_events GROUP BY hour_ts, d, dow, hod
),
daily AS (
  SELECT d, any(dow) AS dow, sum(requests) AS req_total, sum(revenue) AS rev_total
  FROM hourly GROUP BY d
),
daily_trend AS (
  SELECT d, dow, req_total, rev_total,
    avg(req_total) OVER (ORDER BY d ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING) AS req_trend,
    avg(rev_total) OVER (ORDER BY d ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING) AS rev_trend,
    count(req_total) OVER (ORDER BY d ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING) AS trend_n
  FROM daily
),
values_unpivoted AS (
  SELECT h.hour_ts, h.d, h.dow, h.hod, 'requests' AS metric, h.requests / dt.req_trend AS value
  FROM hourly h INNER JOIN daily_trend dt ON h.d = dt.d WHERE dt.trend_n >= 2
  UNION ALL
  SELECT h.hour_ts, h.d, h.dow, h.hod, 'total_revenue', h.revenue / dt.rev_trend
  FROM hourly h INNER JOIN daily_trend dt ON h.d = dt.d WHERE dt.trend_n >= 2
  UNION ALL SELECT hour_ts, d, dow, hod, 'fill_rate', fills/requests FROM hourly
  UNION ALL SELECT hour_ts, d, dow, hod, 'render_rate', impressions/fills FROM hourly
  UNION ALL SELECT hour_ts, d, dow, hod, 'ctr', clicks/impressions FROM hourly
  UNION ALL SELECT hour_ts, d, dow, hod, 'ecpm', revenue/impressions*1000 FROM hourly
  UNION ALL SELECT hour_ts, d, dow, hod, 'rpr', revenue/requests FROM hourly
),
cell_mean AS (SELECT metric, dow, hod, avg(value) AS ca FROM values_unpivoted GROUP BY metric, dow, hod),
noise AS (
  SELECT u.metric AS metric, varSamp(u.value - cm.ca) AS v
  FROM values_unpivoted u JOIN cell_mean cm USING (metric, dow, hod) GROUP BY u.metric
),
seasonal AS (
  SELECT *,
    avg(value) OVER w AS mean_v, varSamp(value) OVER w AS var_v, count(value) OVER w AS baseline_n
  FROM values_unpivoted
  WINDOW w AS (PARTITION BY metric, dow, hod ORDER BY d ROWS BETWEEN 4 PRECEDING AND 1 PRECEDING)
),
zr AS (
  SELECT s.metric AS metric, s.hour_ts AS hour_ts, s.d AS d, s.dow AS dow, s.hod AS hod,
    s.value AS value, s.mean_v AS mean_v, s.baseline_n AS baseline_n,
    sqrt((s.baseline_n*ifNull(s.var_v,0) + 3*n.v) / (s.baseline_n+3)) AS std_v,
    (s.value - s.mean_v) / sqrt((s.baseline_n*ifNull(s.var_v,0) + 3*n.v) / (s.baseline_n+3)) AS zr
  FROM seasonal s JOIN noise n USING (metric)
  WHERE s.baseline_n >= 2
)
"""

METRIC_THRESH_VALUES = "SELECT 'requests' AS metric, {t} AS th UNION ALL SELECT 'total_revenue', {t} " \
    "UNION ALL SELECT 'fill_rate', {t} UNION ALL SELECT 'render_rate', {t} " \
    "UNION ALL SELECT 'ctr', {t} UNION ALL SELECT 'ecpm', {t} UNION ALL SELECT 'rpr', {t}"

TREND_THRESH_VALUES = "SELECT 'requests' AS metric, 2.5 AS th UNION ALL SELECT 'total_revenue', 2.1 " \
    "UNION ALL SELECT 'fill_rate', 2.2 UNION ALL SELECT 'render_rate', 4.0 " \
    "UNION ALL SELECT 'ctr', 4.0 UNION ALL SELECT 'ecpm', 3.3 UNION ALL SELECT 'rpr', 3.4"

flags_trend = query_df(ZR_CTE + f"""
, thresholds AS ({TREND_THRESH_VALUES})
SELECT zr.metric AS metric, 'trend_seasonal' AS method, hour_ts, value AS observed,
  mean_v AS baseline_mean, std_v AS baseline_std, zr AS z
FROM zr INNER JOIN thresholds t ON t.metric = zr.metric
WHERE abs(zr) > t.th
ORDER BY metric, hour_ts
""")

flags_prop = query_df("""
WITH hourly AS (
  SELECT toStartOfHour(event_time) AS hour_ts, toDate(event_time) AS d,
    toDayOfWeek(event_time) AS dow, toHour(event_time) AS hod,
    count() AS requests, sum(is_filled) AS fills, sum(is_impression) AS impressions, sum(is_click) AS clicks
  FROM inmobi.ad_events GROUP BY hour_ts, d, dow, hod
),
counts AS (
  SELECT hour_ts, d, dow, hod, fills AS fill_num, requests AS fill_den,
    impressions AS render_num, fills AS render_den, clicks AS click_num, impressions AS click_den
  FROM hourly
),
unpivoted AS (
  SELECT hour_ts, d, dow, hod, 'fill_rate' AS metric, fill_num AS num, fill_den AS den FROM counts
  UNION ALL SELECT hour_ts, d, dow, hod, 'render_rate', render_num, render_den FROM counts
  UNION ALL SELECT hour_ts, d, dow, hod, 'ctr', click_num, click_den FROM counts
),
seasonal AS (
  SELECT *, num/den AS observed_rate, sum(num) OVER w / sum(den) OVER w AS p0, count(num) OVER w AS baseline_n
  FROM unpivoted
  WINDOW w AS (PARTITION BY metric, dow, hod ORDER BY d ROWS BETWEEN 4 PRECEDING AND 1 PRECEDING)
),
thresholds AS (SELECT 'fill_rate' AS metric, 3.0 AS th UNION ALL SELECT 'render_rate', 4.0 UNION ALL SELECT 'ctr', 4.0)
SELECT s.metric AS metric, 'proportion' AS method, hour_ts, observed_rate AS observed,
  p0 AS baseline_mean, sqrt(p0*(1-p0)/den) AS baseline_std,
  (observed_rate-p0) / sqrt(p0*(1-p0)/den) AS z
FROM seasonal s INNER JOIN thresholds t ON t.metric = s.metric
WHERE baseline_n >= 2 AND abs((observed_rate-p0) / sqrt(p0*(1-p0)/den)) > t.th
ORDER BY metric, hour_ts
""")

flags_day = query_df(ZR_CTE + f"""
, day_stats AS (
  SELECT metric, d, count() AS n, avg(zr) AS mean_zr, avg(zr)*sqrt(count()) AS day_t
  FROM zr GROUP BY metric, d HAVING n = 24
),
thresholds AS ({METRIC_THRESH_VALUES.format(t=3.0)})
SELECT x.metric AS metric, 'day_level' AS method, toDateTime(x.d) AS hour_ts,
  x.mean_zr AS observed, 0.0 AS baseline_mean, 0.0 AS baseline_std, x.day_t AS z
FROM day_stats x INNER JOIN thresholds t ON t.metric = x.metric
WHERE abs(x.day_t) > t.th
ORDER BY metric, hour_ts
""")

flags = pd.concat([flags_trend, flags_prop, flags_day], ignore_index=True)
flags["hour_ts"] = pd.to_datetime(flags["hour_ts"])
print(f"{len(flags)} raw flags across 3 active methods:")
flags.groupby("method").size()

349 raw flags across 3 active methods:


method
day_level          42
proportion         99
trend_seasonal    208
dtype: int64

In [22]:
# Every metric can qualify through either cadence; the source column records which path fired.
DUAL_CADENCE_METRICS = set(METRICS)
REPORTING_ONLY_METRICS = {"ctr", "render_rate"}
INCIDENT_ENABLED_METRICS = DUAL_CADENCE_METRICS - REPORTING_ONLY_METRICS

adverse = flags[flags["z"] < 0].copy()
informational_upticks = flags[flags["z"] > 0].copy()
candidates = []

# Each adverse hour qualifies on its own; method agreement is retained as evidence.
hourly_flags = adverse[adverse["method"] != "day_level"]
for (metric, hour), evidence in hourly_flags.groupby(["metric", "hour_ts"]):
    if metric not in INCIDENT_ENABLED_METRICS:
        continue
    candidates.append({"metric": metric, "start": hour, "end": hour, "source": "hourly",
                       "methods": set(evidence["method"]), "evidence": evidence})

# Each adverse completed day qualifies on its own and represents its full 24-hour interval.
daily_flags = adverse[adverse["method"] == "day_level"].copy()
daily_flags["day"] = daily_flags["hour_ts"].dt.normalize()
for metric, grp in daily_flags.groupby("metric"):
    if metric not in INCIDENT_ENABLED_METRICS:
        continue
    for day, _ in grp.groupby("day"):
        start = day
        end = day + pd.Timedelta(hours=23)
        evidence = adverse[(adverse["metric"] == metric) & adverse["hour_ts"].between(start, end)]
        candidates.append({"metric": metric, "start": start, "end": end, "source": "daily",
                           "methods": set(evidence["method"]), "evidence": evidence})

# Merge only already-qualified candidates for the same metric and adverse direction.
incident_windows = []
for metric in sorted({c["metric"] for c in candidates}):
    spans = sorted((c for c in candidates if c["metric"] == metric), key=lambda c: c["start"])
    for span in spans:
        if incident_windows and incident_windows[-1]["metric"] == metric and span["start"] <= incident_windows[-1]["end"] + pd.Timedelta(hours=1):
            cur = incident_windows[-1]
            cur["end"] = max(cur["end"], span["end"])
            cur["methods"] |= span["methods"]
            cur["sources"].add(span["source"])
            cur["evidence"] = pd.concat([cur["evidence"], span["evidence"]]).drop_duplicates(["metric", "method", "hour_ts"])
        else:
            incident_windows.append({**span, "sources": {span["source"]}})

def metric_readout(metric, start, end):
    win = df[df["hour"].between(start, end)].copy()
    expected = []
    for hour in win["hour"]:
        prior = df[(df["hour"] < hour) & (df["hour"] >= hour - pd.Timedelta(weeks=4))
                   & (df["hour"].dt.dayofweek == hour.dayofweek) & (df["hour"].dt.hour == hour.hour)]
        expected.append(prior[metric].mean())
    return win[metric].mean(), pd.Series(expected).mean()

incidents = []
for w in incident_windows:
    avg_observed, avg_baseline = metric_readout(w["metric"], w["start"], w["end"])
    evidence = w["evidence"]
    incidents.append({
        "metric": w["metric"], "start": w["start"], "end": w["end"],
        "duration_h": (w["end"] - w["start"]).total_seconds() / 3600 + 1,
        "cadence": "+".join(sorted(w["sources"])), "methods": sorted(w["methods"]),
        "n_methods": len(w["methods"]), "avg_observed": avg_observed, "avg_baseline": avg_baseline,
        "avg_z": evidence.groupby("method")["z"].mean().to_dict(),
    })
incidents = pd.DataFrame(incidents).sort_values(["metric", "start"]).reset_index(drop=True)
incidents["pct_delta"] = (incidents["avg_observed"] - incidents["avg_baseline"]) / incidents["avg_baseline"]

print(f"{len(flags)} raw two-sided flags ({len(informational_upticks)} positive informational) -> "
      f"{len(candidates)} grain-qualified adverse units -> {len(incidents)} grouped adverse windows (incident candidates).")
incidents[["metric", "start", "end", "duration_h", "cadence", "methods", "n_methods", "pct_delta"]]

349 raw two-sided flags (63 positive informational) -> 207 grain-qualified adverse units -> 11 grouped adverse windows (incident candidates).


,metric,start,end,duration_h,cadence,methods,n_methods,pct_delta
0,ecpm,2026-06-15 03:00:00,2026-06-15 03:00:00,1.0,hourly,[trend_seasonal],1,-0.037779
1,ecpm,2026-06-17 01:00:00,2026-06-17 01:00:00,1.0,hourly,[trend_seasonal],1,-0.037114
2,ecpm,2026-06-19 00:00:00,2026-06-22 23:00:00,96.0,daily+hourly,"[day_level, trend_seasonal]",2,-0.024338
3,fill_rate,2026-06-18 13:00:00,2026-06-18 13:00:00,1.0,hourly,[proportion],1,-0.020304
4,fill_rate,2026-06-19 02:00:00,2026-06-19 02:00:00,1.0,hourly,[proportion],1,-0.017982
5,fill_rate,2026-06-23 00:00:00,2026-06-25 23:00:00,72.0,daily+hourly,"[day_level, proportion, trend_seasonal]",3,-0.043824
6,fill_rate,2026-06-28 00:00:00,2026-06-29 23:00:00,48.0,daily+hourly,"[day_level, proportion, trend_seasonal]",3,-0.011015
7,requests,2026-06-21 00:00:00,2026-06-21 23:00:00,24.0,daily+hourly,"[day_level, trend_seasonal]",2,-0.434945
8,rpr,2026-06-15 03:00:00,2026-06-15 03:00:00,1.0,hourly,[trend_seasonal],1,-0.050909
9,rpr,2026-06-19 00:00:00,2026-06-25 23:00:00,168.0,daily+hourly,"[day_level, trend_seasonal]",2,-0.032644


## Segment attribution — which dimension/segment is most likely responsible

For each incident, find the single segment most likely driving it, per `PLAN.md`'s Tier-1 design: "rank every dimension independently (`ad_format, category, tier, vertical, campaign_type, region, country, device_model, os_version`) by contribution to the delta."

**Method, per incident:**
1. Restrict to the same `(day-of-week, hour-of-day)` cells the incident window covers, across the whole dataset — `in_window` rows are the incident itself, everything else at those same cells is the baseline (same idea as `trend_seasonal`, just scoped to a dimension breakdown instead of the whole-metric total).
2. For every dimension, compute each segment's own movement: a **% volume change** (requests/revenue) or a **rate-point change** (fill_rate/render_rate/ctr/ecpm/rpr, computed `sum/sum` within the segment, never per-row averaged).
3. **Lift = segment's own movement ÷ the overall metric's movement.** `lift ≈ 1` means the segment moved by about the same amount as everything else (proportional to its size — not a localized cause, just a big segment). `lift >> 1` means the segment moved *disproportionately* more than the metric as a whole — that's the actual localization signal, not raw segment size. Segments under 3% of baseline volume are dropped (a tiny segment can show a wild % swing from noise alone).
4. `vertical`/`campaign_type` are excluded for `requests` and `fill_rate` — both come from the `advertisers` join, which is only populated on already-filled rows, so they can't meaningfully explain *why* something didn't fill or how many requests arrived in the first place.

An incident is called **localized** to a segment when its top lift clears **2×** (moved at least twice as much as the metric overall); otherwise it's reported as **broad-based** — no single segment stands out, the whole metric moved together (which is itself a useful, honest finding, not a failure to find a cause).

In [23]:
SEG_DIMS = ["ad_format", "category", "publisher_tier", "vertical", "campaign_type",
            "region", "country", "device_model", "os_version"]
NUM_DEN = {
    "fill_rate": ("fills", "requests"), "render_rate": ("impressions", "fills"),
    "ctr": ("clicks", "impressions"), "ecpm": ("revenue", "impressions"), "rpr": ("revenue", "requests"),
}
VOLUME_METRICS = {"requests": "requests", "total_revenue": "revenue"}
EXCLUDE_DIMS = {"requests": {"vertical", "campaign_type"}, "fill_rate": {"vertical", "campaign_type"}}
MIN_WEIGHT = 0.03
LOCALIZED_LIFT_THRESHOLD = 2.0


def attribute_segment(metric, start, end, overall_pct_delta):
    """Rank every dimension's segments by lift (own movement / overall movement) for one incident."""
    hours = pd.date_range(start, end, freq="h")
    dow_hod_pairs = sorted(set((h.dayofweek + 1, h.hour) for h in hours))
    pairs_sql = ", ".join(f"({d},{h})" for d, h in dow_hod_pairs)
    union = "\nUNION ALL\n".join(
        f"SELECT '{name}' AS dimension, {name} AS segment, in_window, toStartOfHour(event_time) AS hour_ts, "
        f"count() AS requests, sum(is_filled) AS fills, sum(is_impression) AS impressions, "
        f"sum(is_click) AS clicks, sum(revenue) AS revenue FROM events GROUP BY dimension, segment, in_window, hour_ts"
        for name in SEG_DIMS
    )
    raw = query_df(f"""
    WITH events AS (
      SELECT e.event_time AS event_time,
        (e.event_time >= toDateTime('{start}') AND e.event_time < toDateTime('{end}') + INTERVAL 1 HOUR) AS in_window,
        e.is_filled AS is_filled, e.is_impression AS is_impression, e.is_click AS is_click, e.revenue AS revenue,
        e.ad_format AS ad_format,
        a.category AS category, a.publisher_tier AS publisher_tier,
        adv.vertical AS vertical, adv.campaign_type AS campaign_type,
        g.region AS region, g.country AS country, g.device_model AS device_model, g.os_version AS os_version
      FROM inmobi.ad_events e
      LEFT JOIN inmobi.apps a ON e.app_id = a.app_id
      LEFT JOIN inmobi.advertisers adv ON e.advertiser_id = adv.advertiser_id
      LEFT JOIN inmobi.geo_device g ON e.geo_device_id = g.geo_device_id
      WHERE (toDayOfWeek(e.event_time), toHour(e.event_time)) IN ({pairs_sql})
    )
    {union}
    """)
    window_hours = len(hours)
    baseline_hours = raw[raw["in_window"] == 0]["hour_ts"].nunique()

    agg = raw.groupby(["dimension", "segment", "in_window"]).agg(
        requests=("requests", "sum"), fills=("fills", "sum"), impressions=("impressions", "sum"),
        clicks=("clicks", "sum"), revenue=("revenue", "sum"),
    ).reset_index()
    piv = agg.pivot_table(index=["dimension", "segment"], columns="in_window",
                           values=["requests", "fills", "impressions", "clicks", "revenue"], fill_value=0)
    piv.columns = [f"{a}_{b}" for a, b in piv.columns]
    piv = piv.reset_index()
    excl = EXCLUDE_DIMS.get(metric, set())
    piv = piv[(piv["segment"] != "") & (~piv["dimension"].isin(excl))]

    if metric in VOLUME_METRICS:
        col = VOLUME_METRICS[metric]
        piv["baseline_scaled"] = piv[f"{col}_0"] / baseline_hours * window_hours
        piv["weight"] = piv[f"{col}_0"] / piv.groupby("dimension")[f"{col}_0"].transform("sum")
        piv["seg_pct_delta"] = (piv[f"{col}_1"] - piv["baseline_scaled"]) / piv["baseline_scaled"].replace(0, pd.NA)
        piv["lift"] = piv["seg_pct_delta"] / overall_pct_delta
    else:
        num, den = NUM_DEN[metric]
        piv["rate_0"] = piv[f"{num}_0"] / piv[f"{den}_0"].replace(0, pd.NA)
        piv["rate_1"] = piv[f"{num}_1"] / piv[f"{den}_1"].replace(0, pd.NA)
        piv["seg_pct_delta"] = piv["rate_1"] - piv["rate_0"]
        piv["weight"] = piv[f"{den}_0"] / piv.groupby("dimension")[f"{den}_0"].transform("sum")
        dim_total_delta = piv.groupby("dimension").apply(
            lambda g: (g["weight"] * g["seg_pct_delta"]).sum(), include_groups=False
        )
        piv = piv.join(dim_total_delta.rename("dim_total_delta"), on="dimension")
        piv["lift"] = piv["seg_pct_delta"] / piv["dim_total_delta"]

    piv = piv[piv["weight"] >= MIN_WEIGHT].dropna(subset=["lift"])
    return piv.reindex(piv["lift"].abs().sort_values(ascending=False).index)[
        ["dimension", "segment", "weight", "seg_pct_delta", "lift"]
    ].reset_index(drop=True)


attribution_results = []
for _, inc in incidents.iterrows():
    ranked = attribute_segment(inc["metric"], inc["start"], inc["end"], inc["pct_delta"])
    top = ranked.iloc[0] if len(ranked) else None
    attribution_results.append({
        "metric": inc["metric"], "start": inc["start"], "end": inc["end"],
        "top_dimension": top["dimension"] if top is not None else None,
        "top_segment": top["segment"] if top is not None else None,
        "top_weight": top["weight"] if top is not None else None,
        "top_lift": top["lift"] if top is not None else None,
        "localized": bool(top is not None and abs(top["lift"]) >= LOCALIZED_LIFT_THRESHOLD),
        "ranked": ranked,
    })
attribution_df = pd.DataFrame(attribution_results)
incidents = incidents.merge(
    attribution_df[["metric", "start", "end", "top_dimension", "top_segment", "top_lift", "localized"]],
    on=["metric", "start", "end"],
)
print(f"{incidents['localized'].sum()} / {len(incidents)} incidents localized to a single segment (lift >= {LOCALIZED_LIFT_THRESHOLD}x)")
incidents[["metric", "start", "end", "top_dimension", "top_segment", "top_lift", "localized"]]

9 / 11 incidents localized to a single segment (lift >= 2.0x)


,metric,start,end,top_dimension,top_segment,top_lift,localized
0,ecpm,2026-06-15 03:00:00,2026-06-15 03:00:00,ad_format,video,5.326119,True
1,ecpm,2026-06-17 01:00:00,2026-06-17 01:00:00,ad_format,interstitial,10.960049,True
2,ecpm,2026-06-19 00:00:00,2026-06-22 23:00:00,category,finance,14.000658,True
3,fill_rate,2026-06-18 13:00:00,2026-06-18 13:00:00,os_version,Android 15,-16.395498,True
4,fill_rate,2026-06-19 02:00:00,2026-06-19 02:00:00,country,FR,-6.170064,True
5,fill_rate,2026-06-23 00:00:00,2026-06-25 23:00:00,os_version,Android 15,10.601638,True
6,fill_rate,2026-06-28 00:00:00,2026-06-29 23:00:00,os_version,iOS 18.1,11.116519,True
7,requests,2026-06-21 00:00:00,2026-06-21 23:00:00,category,news,1.065559,False
8,rpr,2026-06-15 03:00:00,2026-06-15 03:00:00,ad_format,video,6.088490,True
9,rpr,2026-06-19 00:00:00,2026-06-25 23:00:00,os_version,Android 15,7.027790,True


### Summary table — qualified incident candidates, ranked by confidence then severity

In [24]:
SUMMARY_METRIC_LABEL = {
    "requests": "Requests", "fill_rate": "Fill rate", "render_rate": "Render rate",
    "ctr": "CTR", "total_revenue": "Revenue", "ecpm": "eCPM", "rpr": "RPR",
}

summary = incidents.copy()
summary["window"] = summary.apply(
    lambda r: f"{r['start']:%d/%m/%Y %H:%M} → {r['end']:%d/%m/%Y %H:%M}", axis=1
)
summary["direction"] = summary["pct_delta"].apply(lambda x: "▲ rose" if x > 0 else "▼ fell")
summary["% delta"] = summary["pct_delta"].apply(lambda x: f"{x:+.1%}")
summary["methods"] = summary["methods"].apply(", ".join)
summary["confidence"] = summary["n_methods"].apply(lambda n: "high" if n >= 2 else "medium")
summary["severity"] = summary["duration_h"] * summary["n_methods"]
summary["attributed segment"] = summary.apply(
    lambda r: f"{r['top_dimension']}={r['top_segment']} ({r['top_lift']:.1f}x)" if r["localized"]
    else "broad-based (no segment stands out)",
    axis=1,
)

summary = summary.sort_values(["confidence", "severity"], ascending=[True, False])
summary["metric"] = summary["metric"].map(SUMMARY_METRIC_LABEL)

summary[["metric", "window", "duration_h", "direction", "% delta", "methods", "n_methods",
         "confidence", "attributed segment"]].rename(
    columns={"duration_h": "duration (h)", "n_methods": "# methods agreeing"}
).reset_index(drop=True)

,metric,window,duration (h),direction,% delta,methods,# methods agreeing,confidence,attributed segment
0,RPR,19/06/2026 00:00 → 25/06/2026 23:00,168.0,▼ fell,-3.3%,"day_level, trend_seasonal",2,high,os_version=Android 15 (7.0x)
1,Fill rate,23/06/2026 00:00 → 25/06/2026 23:00,72.0,▼ fell,-4.4%,"day_level, proportion, trend_seasonal",3,high,os_version=Android 15 (10.6x)
2,eCPM,19/06/2026 00:00 → 22/06/2026 23:00,96.0,▼ fell,-2.4%,"day_level, trend_seasonal",2,high,category=finance (14.0x)
3,Fill rate,28/06/2026 00:00 → 29/06/2026 23:00,48.0,▼ fell,-1.1%,"day_level, proportion, trend_seasonal",3,high,os_version=iOS 18.1 (11.1x)
4,Requests,21/06/2026 00:00 → 21/06/2026 23:00,24.0,▼ fell,-43.5%,"day_level, trend_seasonal",2,high,broad-based (no segment stands out)
5,Revenue,21/06/2026 00:00 → 21/06/2026 23:00,24.0,▼ fell,-44.8%,"day_level, trend_seasonal",2,high,broad-based (no segment stands out)
6,eCPM,15/06/2026 03:00 → 15/06/2026 03:00,1.0,▼ fell,-3.8%,trend_seasonal,1,medium,ad_format=video (5.3x)
7,eCPM,17/06/2026 01:00 → 17/06/2026 01:00,1.0,▼ fell,-3.7%,trend_seasonal,1,medium,ad_format=interstitial (11.0x)
8,Fill rate,18/06/2026 13:00 → 18/06/2026 13:00,1.0,▼ fell,-2.0%,proportion,1,medium,os_version=Android 15 (-16.4x)
9,Fill rate,19/06/2026 02:00 → 19/06/2026 02:00,1.0,▼ fell,-1.8%,proportion,1,medium,country=FR (-6.2x)


In [25]:
from IPython.display import Markdown, display

METRIC_LABEL = {
    "requests": "Requests", "fill_rate": "Fill rate", "render_rate": "Render rate",
    "ctr": "CTR", "total_revenue": "Revenue", "ecpm": "eCPM", "rpr": "RPR",
}
METRIC_FMT = {
    "requests": "{:,.0f}", "fill_rate": "{:.2%}", "render_rate": "{:.2%}",
    "ctr": "{:.3%}", "total_revenue": "${:,.2f}", "ecpm": "${:.3f}", "rpr": "${:.5f}",
}
METRIC_STORY = {
    "requests": "raw ad-request volume — a shift here means fewer or more opportunities entered the funnel, upstream of any monetization logic (fill, price, engagement)",
    "fill_rate": "what fraction of requests the network could monetize — a drop points at a supply/demand mismatch (fewer advertisers bidding on that inventory) or a fill-side integration issue, a rise at the opposite",
    "render_rate": "the fill-to-impression leak — usually a technical/client-side issue (slow creative load, user navigating away, an SDK bug) rather than a marketplace demand issue, since the ad was already won by the time this step happens",
    "ctr": "ad engagement quality — a diagnostic signal on creative/targeting fit, not a direct revenue driver in this CPM-priced dataset",
    "total_revenue": "the bottom-line dollar outcome — the net effect of whatever moved upstream in requests, fill rate, or eCPM",
    "ecpm": "price per 1,000 impressions — a drop points at advertiser demand shifting toward cheaper campaigns, an auction/pricing-floor misconfiguration, or a high-value advertiser vertical churning",
    "rpr": "all-in revenue efficiency per request — mechanically inherits movement from requests, fill rate, and eCPM, so it should be explained by one of those rather than treated as an independent cause",
}
METHOD_LABEL = {
    "trend_seasonal": "trend_seasonal (trailing seasonal baseline, shrunk variance)",
    "proportion": "proportion (two-proportion z-test on the hour's own counts)",
    "day_level": "day_level (whole-day pooled t-test)",
}

for _, row in incidents.iterrows():
    m = row["metric"]
    direction = "rose" if row["pct_delta"] > 0 else "fell"
    day_span = f"{row['start']:%d/%m/%Y %H:%M} → {row['end']:%d/%m/%Y %H:%M}"
    fmt = METRIC_FMT[m]
    z_lines = "\n".join(f"| z ({meth}) | {z:+.1f} |" for meth, z in row["avg_z"].items())
    confidence = ("high — multiple independent methods agree" if row["n_methods"] >= 2
                  else f"medium — one method qualified through the {row['cadence']} persistence rule")

    if row["localized"]:
        segment_line = (
            f"**Segment attribution:** localized to **`{row['top_dimension']}` = `{row['top_segment']}`** — "
            f"this segment moved **{row['top_lift']:.1f}x** more than the metric as a whole "
            f"(own change {row['top_lift']*row['pct_delta']:+.1%} vs overall {row['pct_delta']:+.1%}), "
            f"well past the 2x bar for calling it a likely driver rather than a side effect of the broader move."
        )
    else:
        segment_line = (
            "**Segment attribution:** broad-based — no single segment across `ad_format`, `category`, "
            "`publisher_tier`, `vertical`, `campaign_type`, `region`, `country`, `device_model`, or `os_version` "
            "moved disproportionately more than the metric overall; every segment checked moved roughly in line "
            "with the aggregate, so this looks systemic rather than isolated to one slice of inventory."
        )

    md = f"""### {METRIC_LABEL[m]} — {direction} {abs(row['pct_delta']):.1%}, {day_span} ({row['duration_h']:.0f}h)

**Flagged by:** {", ".join(METHOD_LABEL[meth] for meth in row["methods"])} — confidence: **{confidence}**

{segment_line}

| Evidence | Value |
|---|---|
| Window | {day_span} — {row['duration_h']:.0f}h |
| Observed (avg over window) | {fmt.format(row['avg_observed'])} |
| Baseline (trailing seasonal, same hour-of-week) | {fmt.format(row['avg_baseline'])} |
| % delta vs baseline | {row['pct_delta']:+.1%} |
{z_lines}

**Why this qualifies as an incident candidate:** this adverse statistical unit qualified through the **{row['cadence']} grain**, backed by {row['n_methods']} method(s) — {", ".join(row["methods"])}. Every metric is evaluated independently at hourly and completed-day grain; crossing the applicable detector threshold is sufficient. Adjacent qualified units are grouped only to describe the candidate window, not as an additional persistence requirement. Positive deviations remain informational. Confirmation and paging belong to downstream triage. This metric measures {METRIC_STORY[m]}.
"""
    display(Markdown(md))

### eCPM — fell 3.8%, 15/06/2026 03:00 → 15/06/2026 03:00 (1h)

**Flagged by:** trend_seasonal (trailing seasonal baseline, shrunk variance) — confidence: **medium — one method qualified through the hourly persistence rule**

**Segment attribution:** localized to **`ad_format` = `video`** — this segment moved **5.3x** more than the metric as a whole (own change -20.1% vs overall -3.8%), well past the 2x bar for calling it a likely driver rather than a side effect of the broader move.

| Evidence | Value |
|---|---|
| Window | 15/06/2026 03:00 → 15/06/2026 03:00 — 1h |
| Observed (avg over window) | $2.405 |
| Baseline (trailing seasonal, same hour-of-week) | $2.500 |
| % delta vs baseline | -3.8% |
| z (trend_seasonal) | -4.1 |

**Why this qualifies as an incident candidate:** this adverse statistical unit qualified through the **hourly grain**, backed by 1 method(s) — trend_seasonal. Every metric is evaluated independently at hourly and completed-day grain; crossing the applicable detector threshold is sufficient. Adjacent qualified units are grouped only to describe the candidate window, not as an additional persistence requirement. Positive deviations remain informational. Confirmation and paging belong to downstream triage. This metric measures price per 1,000 impressions — a drop points at advertiser demand shifting toward cheaper campaigns, an auction/pricing-floor misconfiguration, or a high-value advertiser vertical churning.


### eCPM — fell 3.7%, 17/06/2026 01:00 → 17/06/2026 01:00 (1h)

**Flagged by:** trend_seasonal (trailing seasonal baseline, shrunk variance) — confidence: **medium — one method qualified through the hourly persistence rule**

**Segment attribution:** localized to **`ad_format` = `interstitial`** — this segment moved **11.0x** more than the metric as a whole (own change -40.7% vs overall -3.7%), well past the 2x bar for calling it a likely driver rather than a side effect of the broader move.

| Evidence | Value |
|---|---|
| Window | 17/06/2026 01:00 → 17/06/2026 01:00 — 1h |
| Observed (avg over window) | $2.411 |
| Baseline (trailing seasonal, same hour-of-week) | $2.504 |
| % delta vs baseline | -3.7% |
| z (trend_seasonal) | -3.6 |

**Why this qualifies as an incident candidate:** this adverse statistical unit qualified through the **hourly grain**, backed by 1 method(s) — trend_seasonal. Every metric is evaluated independently at hourly and completed-day grain; crossing the applicable detector threshold is sufficient. Adjacent qualified units are grouped only to describe the candidate window, not as an additional persistence requirement. Positive deviations remain informational. Confirmation and paging belong to downstream triage. This metric measures price per 1,000 impressions — a drop points at advertiser demand shifting toward cheaper campaigns, an auction/pricing-floor misconfiguration, or a high-value advertiser vertical churning.


### eCPM — fell 2.4%, 19/06/2026 00:00 → 22/06/2026 23:00 (96h)

**Flagged by:** day_level (whole-day pooled t-test), trend_seasonal (trailing seasonal baseline, shrunk variance) — confidence: **high — multiple independent methods agree**

**Segment attribution:** localized to **`category` = `finance`** — this segment moved **14.0x** more than the metric as a whole (own change -34.1% vs overall -2.4%), well past the 2x bar for calling it a likely driver rather than a side effect of the broader move.

| Evidence | Value |
|---|---|
| Window | 19/06/2026 00:00 → 22/06/2026 23:00 — 96h |
| Observed (avg over window) | $2.415 |
| Baseline (trailing seasonal, same hour-of-week) | $2.475 |
| % delta vs baseline | -2.4% |
| z (day_level) | -10.9 |
| z (trend_seasonal) | -4.1 |

**Why this qualifies as an incident candidate:** this adverse statistical unit qualified through the **daily+hourly grain**, backed by 2 method(s) — day_level, trend_seasonal. Every metric is evaluated independently at hourly and completed-day grain; crossing the applicable detector threshold is sufficient. Adjacent qualified units are grouped only to describe the candidate window, not as an additional persistence requirement. Positive deviations remain informational. Confirmation and paging belong to downstream triage. This metric measures price per 1,000 impressions — a drop points at advertiser demand shifting toward cheaper campaigns, an auction/pricing-floor misconfiguration, or a high-value advertiser vertical churning.


### Fill rate — fell 2.0%, 18/06/2026 13:00 → 18/06/2026 13:00 (1h)

**Flagged by:** proportion (two-proportion z-test on the hour's own counts) — confidence: **medium — one method qualified through the hourly persistence rule**

**Segment attribution:** localized to **`os_version` = `Android 15`** — this segment moved **-16.4x** more than the metric as a whole (own change +33.3% vs overall -2.0%), well past the 2x bar for calling it a likely driver rather than a side effect of the broader move.

| Evidence | Value |
|---|---|
| Window | 18/06/2026 13:00 → 18/06/2026 13:00 — 1h |
| Observed (avg over window) | 77.33% |
| Baseline (trailing seasonal, same hour-of-week) | 78.93% |
| % delta vs baseline | -2.0% |
| z (proportion) | -4.7 |

**Why this qualifies as an incident candidate:** this adverse statistical unit qualified through the **hourly grain**, backed by 1 method(s) — proportion. Every metric is evaluated independently at hourly and completed-day grain; crossing the applicable detector threshold is sufficient. Adjacent qualified units are grouped only to describe the candidate window, not as an additional persistence requirement. Positive deviations remain informational. Confirmation and paging belong to downstream triage. This metric measures what fraction of requests the network could monetize — a drop points at a supply/demand mismatch (fewer advertisers bidding on that inventory) or a fill-side integration issue, a rise at the opposite.


### Fill rate — fell 1.8%, 19/06/2026 02:00 → 19/06/2026 02:00 (1h)

**Flagged by:** proportion (two-proportion z-test on the hour's own counts) — confidence: **medium — one method qualified through the hourly persistence rule**

**Segment attribution:** localized to **`country` = `FR`** — this segment moved **-6.2x** more than the metric as a whole (own change +11.1% vs overall -1.8%), well past the 2x bar for calling it a likely driver rather than a side effect of the broader move.

| Evidence | Value |
|---|---|
| Window | 19/06/2026 02:00 → 19/06/2026 02:00 — 1h |
| Observed (avg over window) | 77.45% |
| Baseline (trailing seasonal, same hour-of-week) | 78.86% |
| % delta vs baseline | -1.8% |
| z (proportion) | -3.2 |

**Why this qualifies as an incident candidate:** this adverse statistical unit qualified through the **hourly grain**, backed by 1 method(s) — proportion. Every metric is evaluated independently at hourly and completed-day grain; crossing the applicable detector threshold is sufficient. Adjacent qualified units are grouped only to describe the candidate window, not as an additional persistence requirement. Positive deviations remain informational. Confirmation and paging belong to downstream triage. This metric measures what fraction of requests the network could monetize — a drop points at a supply/demand mismatch (fewer advertisers bidding on that inventory) or a fill-side integration issue, a rise at the opposite.


...

### Fill rate — fell 1.1%, 28/06/2026 00:00 → 29/06/2026 23:00 (48h)

**Flagged by:** day_level (whole-day pooled t-test), proportion (two-proportion z-test on the hour's own counts), trend_seasonal (trailing seasonal baseline, shrunk variance) — confidence: **high — multiple independent methods agree**

**Segment attribution:** localized to **`os_version` = `iOS 18.1`** — this segment moved **11.1x** more than the metric as a whole (own change -12.2% vs overall -1.1%), well past the 2x bar for calling it a likely driver rather than a side effect of the broader move.

| Evidence | Value |
|---|---|
| Window | 28/06/2026 00:00 → 29/06/2026 23:00 — 48h |
| Observed (avg over window) | 77.64% |
| Baseline (trailing seasonal, same hour-of-week) | 78.51% |
| % delta vs baseline | -1.1% |
| z (day_level) | -5.9 |
| z (proportion) | -3.6 |
| z (trend_seasonal) | -2.3 |

**Why this qualifies as an incident candidate:** this adverse statistical unit qualified through the **daily+hourly grain**, backed by 3 method(s) — day_level, proportion, trend_seasonal. Every metric is evaluated independently at hourly and completed-day grain; crossing the applicable detector threshold is sufficient. Adjacent qualified units are grouped only to describe the candidate window, not as an additional persistence requirement. Positive deviations remain informational. Confirmation and paging belong to downstream triage. This metric measures what fraction of requests the network could monetize — a drop points at a supply/demand mismatch (fewer advertisers bidding on that inventory) or a fill-side integration issue, a rise at the opposite.


### Requests — fell 43.5%, 21/06/2026 00:00 → 21/06/2026 23:00 (24h)

**Flagged by:** day_level (whole-day pooled t-test), trend_seasonal (trailing seasonal baseline, shrunk variance) — confidence: **high — multiple independent methods agree**

**Segment attribution:** broad-based — no single segment across `ad_format`, `category`, `publisher_tier`, `vertical`, `campaign_type`, `region`, `country`, `device_model`, or `os_version` moved disproportionately more than the metric overall; every segment checked moved roughly in line with the aggregate, so this looks systemic rather than isolated to one slice of inventory.

| Evidence | Value |
|---|---|
| Window | 21/06/2026 00:00 → 21/06/2026 23:00 — 24h |
| Observed (avg over window) | 5,252 |
| Baseline (trailing seasonal, same hour-of-week) | 9,295 |
| % delta vs baseline | -43.5% |
| z (day_level) | -32.6 |
| z (trend_seasonal) | -6.6 |

**Why this qualifies as an incident candidate:** this adverse statistical unit qualified through the **daily+hourly grain**, backed by 2 method(s) — day_level, trend_seasonal. Every metric is evaluated independently at hourly and completed-day grain; crossing the applicable detector threshold is sufficient. Adjacent qualified units are grouped only to describe the candidate window, not as an additional persistence requirement. Positive deviations remain informational. Confirmation and paging belong to downstream triage. This metric measures raw ad-request volume — a shift here means fewer or more opportunities entered the funnel, upstream of any monetization logic (fill, price, engagement).


### RPR — fell 5.1%, 15/06/2026 03:00 → 15/06/2026 03:00 (1h)

**Flagged by:** trend_seasonal (trailing seasonal baseline, shrunk variance) — confidence: **medium — one method qualified through the hourly persistence rule**

**Segment attribution:** localized to **`ad_format` = `video`** — this segment moved **6.1x** more than the metric as a whole (own change -31.0% vs overall -5.1%), well past the 2x bar for calling it a likely driver rather than a side effect of the broader move.

| Evidence | Value |
|---|---|
| Window | 15/06/2026 03:00 → 15/06/2026 03:00 — 1h |
| Observed (avg over window) | $0.00184 |
| Baseline (trailing seasonal, same hour-of-week) | $0.00194 |
| % delta vs baseline | -5.1% |
| z (trend_seasonal) | -4.0 |

**Why this qualifies as an incident candidate:** this adverse statistical unit qualified through the **hourly grain**, backed by 1 method(s) — trend_seasonal. Every metric is evaluated independently at hourly and completed-day grain; crossing the applicable detector threshold is sufficient. Adjacent qualified units are grouped only to describe the candidate window, not as an additional persistence requirement. Positive deviations remain informational. Confirmation and paging belong to downstream triage. This metric measures all-in revenue efficiency per request — mechanically inherits movement from requests, fill rate, and eCPM, so it should be explained by one of those rather than treated as an independent cause.


### RPR — fell 3.3%, 19/06/2026 00:00 → 25/06/2026 23:00 (168h)

**Flagged by:** day_level (whole-day pooled t-test), trend_seasonal (trailing seasonal baseline, shrunk variance) — confidence: **high — multiple independent methods agree**

**Segment attribution:** localized to **`os_version` = `Android 15`** — this segment moved **7.0x** more than the metric as a whole (own change -22.9% vs overall -3.3%), well past the 2x bar for calling it a likely driver rather than a side effect of the broader move.

| Evidence | Value |
|---|---|
| Window | 19/06/2026 00:00 → 25/06/2026 23:00 — 168h |
| Observed (avg over window) | $0.00184 |
| Baseline (trailing seasonal, same hour-of-week) | $0.00190 |
| % delta vs baseline | -3.3% |
| z (day_level) | -11.2 |
| z (trend_seasonal) | -4.0 |

**Why this qualifies as an incident candidate:** this adverse statistical unit qualified through the **daily+hourly grain**, backed by 2 method(s) — day_level, trend_seasonal. Every metric is evaluated independently at hourly and completed-day grain; crossing the applicable detector threshold is sufficient. Adjacent qualified units are grouped only to describe the candidate window, not as an additional persistence requirement. Positive deviations remain informational. Confirmation and paging belong to downstream triage. This metric measures all-in revenue efficiency per request — mechanically inherits movement from requests, fill rate, and eCPM, so it should be explained by one of those rather than treated as an independent cause.


### Revenue — fell 44.8%, 21/06/2026 00:00 → 21/06/2026 23:00 (24h)

**Flagged by:** day_level (whole-day pooled t-test), trend_seasonal (trailing seasonal baseline, shrunk variance) — confidence: **high — multiple independent methods agree**

**Segment attribution:** broad-based — no single segment across `ad_format`, `category`, `publisher_tier`, `vertical`, `campaign_type`, `region`, `country`, `device_model`, or `os_version` moved disproportionately more than the metric overall; every segment checked moved roughly in line with the aggregate, so this looks systemic rather than isolated to one slice of inventory.

| Evidence | Value |
|---|---|
| Window | 21/06/2026 00:00 → 21/06/2026 23:00 — 24h |
| Observed (avg over window) | $9.77 |
| Baseline (trailing seasonal, same hour-of-week) | $17.71 |
| % delta vs baseline | -44.8% |
| z (day_level) | -31.6 |
| z (trend_seasonal) | -6.5 |

**Why this qualifies as an incident candidate:** this adverse statistical unit qualified through the **daily+hourly grain**, backed by 2 method(s) — day_level, trend_seasonal. Every metric is evaluated independently at hourly and completed-day grain; crossing the applicable detector threshold is sufficient. Adjacent qualified units are grouped only to describe the candidate window, not as an additional persistence requirement. Positive deviations remain informational. Confirmation and paging belong to downstream triage. This metric measures the bottom-line dollar outcome — the net effect of whatever moved upstream in requests, fill rate, or eCPM.
